# Re processing Cluster level data

In [ ]:
import pandas as pd
import numpy as np
import csv
import gzip
import matplotlib.pyplot as plt

In [ ]:
Subcluster_anno = pd.read_excel("/home/jw3514/Work/data/HumanBrainCellType/subcluster_annotation.xlsx", index_col="Subcluster")
DatDIR = "/home/jw3514/Work/data/HumanBrainCellType/Subcluster_GeneXCell/"

In [ ]:
def processHumanCT_Cluster(cluster, annotation, DatDIR):
    print("Processing {}".format(cluster))
    subclusters = annotation[annotation["Cluster"]==cluster].index.values
    print(subclusters)
    
    dfs = []
    for subcluster in subclusters:
        df = pd.read_csv("{}/{}.csv.gz".format(DatDIR, subcluster), index_col=0)
        dfs.append(df)
    
    # Concatenate all dataframes along columns
    combined_df = pd.concat(dfs, axis=1)
    combined_df.to_csv("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/{}.csv.gz".format(cluster))
    
    # Calculate plain mean for comparison
    gene_means = combined_df.values.mean(axis=1) 
    gene_means = pd.Series(data=gene_means, index=combined_df.index)
    gene_means.to_csv("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_mean.csv".format(cluster))
    
    # Apply log2(x+1) transformation and take mean across cells
    gene_log2_means = np.log2(combined_df.values + 1).mean(axis=1)
    gene_log2_means = pd.Series(data=gene_log2_means, index=combined_df.index)
    gene_log2_means.to_csv("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_log2mean.csv".format(cluster))

        #Gene_Cluster_Mean = pd.Series(data=gene_dat, index=gene_index)
        #Gene_Cluster_Mean.to_csv("/home/jw3514/Work/data/HumanBrainCellType/Subcluster_GeneXCell/SplitCTs/{}.csv".format(cluster))

In [ ]:
def processHumanCT_Cluster_LargeFile(cluster, annotation, DatDIR):
    print("Processing {}".format(cluster))
    subclusters = annotation[annotation["Cluster"]==cluster].index.values
    print(subclusters)
    
    # Initialize dictionaries to store sums and counts
    gene_sums = {}
    gene_counts = {}
    gene_log2_sums = {}
    first_file = True
    
    # Process each subcluster file
    for subcluster in subclusters:
        with gzip.open("{}/{}.csv.gz".format(DatDIR, subcluster), 'rt') as f:
            reader = csv.reader(f)
            header = next(reader) # Skip header
            
            # For first file, initialize gene names
            if first_file:
                gene_names = []
                for row in reader:
                    gene_name = row[0]
                    gene_names.append(gene_name)
                    values = [float(x) for x in row[1:]]
                    gene_sums[gene_name] = sum(values)
                    gene_counts[gene_name] = len(values)
                    gene_log2_sums[gene_name] = sum(np.log2(np.array(values) + 1))
                first_file = False
            else:
                for row in reader:
                    gene_name = row[0]
                    values = [float(x) for x in row[1:]]
                    gene_sums[gene_name] += sum(values)
                    gene_counts[gene_name] += len(values)
                    gene_log2_sums[gene_name] += sum(np.log2(np.array(values) + 1))
    
    # Calculate means and save results
    with open("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_mean.csv".format(cluster), 'w') as f:
        writer = csv.writer(f)
        writer.writerow(['gene','0'])
        for gene in gene_names:
            mean = gene_sums[gene] / gene_counts[gene]
            writer.writerow([gene, mean])
            
    with open("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_log2mean.csv".format(cluster), 'w') as f:
        writer = csv.writer(f)
        writer.writerow(['gene','0']) 
        for gene in gene_names:
            log2_mean = gene_log2_sums[gene] / gene_counts[gene]
            writer.writerow([gene, log2_mean])

In [ ]:
Clusters = Subcluster_anno["Cluster"].unique()
Clusters.sort()

## Aggregate split files into one matrix

In [ ]:
# Load annotation - adjust this path based on your annotation file
Anno = Subcluster_anno.groupby('Cluster').first()  # Get one row per cluster

In [ ]:
Indv_cluster_means = []
Indv_cluster_log2means = []
Missing_Clusters = []
for cluster, row in Anno.iterrows():
    #_subcluster_name = "{}-{}-{}".format(_subcluster_id, _subcluster_ann["Cluster"], _subcluster_ann["Supercluster"])
    try:
        gene_mean_UMI = pd.read_csv(("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_mean.csv".format(cluster)), index_col=0)
        gene_mean_logUMI = pd.read_csv(("/home/jw3514/Work/data/HumanBrainCellType/cluster_GeneXCell/SplitCTs/{}_log2mean.csv".format(cluster)), index_col=0)
        gene_mean_UMI.rename(columns={"0": cluster}, inplace=True)
        gene_mean_logUMI.rename(columns={"0": cluster}, inplace=True)

        Indv_cluster_means.append(gene_mean_UMI)
        Indv_cluster_log2means.append(gene_mean_logUMI)
    except:
        print("{} not found".format(cluster))
        Missing_Clusters.append(cluster)
        
# Make and save cluster Exp Mat
Cluster_Exp_DF = pd.concat(Indv_cluster_means, axis=1)
Cluster_Exp_DF.to_csv("/home/jw3514/Work/data/HumanBrainCellType/cluster_MeanUMI.csv")

Cluster_Exp_DF_log2 = pd.concat(Indv_cluster_log2means, axis=1)
Cluster_Exp_DF_log2.to_csv("/home/jw3514/Work/data/HumanBrainCellType/cluster_MeanLogUMI.csv")

In [ ]:
Cluster_Exp_DF.head(2)

In [ ]:
Cluster_Exp_DF_log2.head(2)

## Remove duplicate genes

In [ ]:
Cluster_Exp_DF = Cluster_Exp_DF.loc[~Cluster_Exp_DF.index.duplicated(keep='first')]

In [ ]:
Cluster_Exp_DF_log2 = Cluster_Exp_DF_log2.loc[~Cluster_Exp_DF_log2.index.duplicated(keep='first')]

## Save final expression matrix

In [ ]:
Cluster_Exp_DF_log2.to_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/Human.Cluster.Log2Mean.Exp.csv")